# Gold Feature Layer — Validation

Validation checks for `txn_features_gold` and `customer_risk_profile_gold` in `workshop_guy_catalog.fe_bar_a0`.

## Feature Descriptions
- **amount** — Raw transaction amount
- **amt_to_monthly_spend** — Transaction amount divided by the customer’s average monthly spend (NULL-safe; NULL when avg_monthly_spend is 0 or NULL)
- **home_country** — Derived from home_region: US-East/US-West→US, EU→UK, APAC→JP, else BR
- **is_cross_border** — 1 if merchant country differs from home_country
- **is_new_device** — 1 if this is the customer’s first-ever transaction on this device_id
- **is_online** — 1 if channel = ‘online’
- **txn_count_1h / txn_count_24h** — Count of the customer’s transactions in the strictly prior 1h / 24h window
- **amt_to_cust_avg_30d** — Transaction amount divided by the customer’s average amount over the strictly prior 30 days
- **secs_since_prev_txn** — Seconds since the customer’s previous transaction
- **hour_of_day** — Hour component (0–23) of the transaction timestamp
- **is_night** — 1 if hour_of_day is between 0 and 5
- **mcc_category** — Merchant category code
- **tenure** — Customer tenure in months
- **risk_tier / risk_segment** — Merchant risk tier and customer risk segment
- **is_fraud** — Label (boolean), passed through unchanged from silver

In [0]:
%sql
-- (a) Fraud rate grouped by is_cross_border, is_new_device, is_online, and amt_to_monthly_spend > 3
-- Rows with these fraud signals should show fraud rates well above the ~0.5% baseline

SELECT
  'is_cross_border' AS signal,
  is_cross_border   AS value,
  COUNT(*)          AS txn_count,
  SUM(CAST(is_fraud AS INT)) AS fraud_count,
  ROUND(100.0 * SUM(CAST(is_fraud AS INT)) / COUNT(*), 4) AS fraud_rate_pct
FROM workshop_guy_catalog.fe_bar_a0.txn_features_gold
GROUP BY is_cross_border

UNION ALL

SELECT
  'is_new_device',
  is_new_device,
  COUNT(*),
  SUM(CAST(is_fraud AS INT)),
  ROUND(100.0 * SUM(CAST(is_fraud AS INT)) / COUNT(*), 4)
FROM workshop_guy_catalog.fe_bar_a0.txn_features_gold
GROUP BY is_new_device

UNION ALL

SELECT
  'is_online',
  is_online,
  COUNT(*),
  SUM(CAST(is_fraud AS INT)),
  ROUND(100.0 * SUM(CAST(is_fraud AS INT)) / COUNT(*), 4)
FROM workshop_guy_catalog.fe_bar_a0.txn_features_gold
GROUP BY is_online

UNION ALL

SELECT
  'amt_to_monthly_spend_gt_3',
  CAST(CASE WHEN amt_to_monthly_spend > 3 THEN 1 ELSE 0 END AS INT),
  COUNT(*),
  SUM(CAST(is_fraud AS INT)),
  ROUND(100.0 * SUM(CAST(is_fraud AS INT)) / COUNT(*), 4)
FROM workshop_guy_catalog.fe_bar_a0.txn_features_gold
GROUP BY CASE WHEN amt_to_monthly_spend > 3 THEN 1 ELSE 0 END

ORDER BY signal, value

signal,value,txn_count,fraud_count,fraud_rate_pct
amt_to_monthly_spend_gt_3,0,457228,8756,1.9150
amt_to_monthly_spend_gt_3,1,7451,1784,23.9431
is_cross_border,0,142230,2378,1.6719
is_cross_border,1,322449,8162,2.5313
is_new_device,0,415854,6630,1.5943
is_new_device,1,48825,3910,8.0082
is_online,0,302115,4806,1.5908
is_online,1,162564,5734,3.5272


In [0]:
%sql
-- (b) Check that amt_to_monthly_spend is NOT mostly NULL (verifies column mask not applied)
--     and MAX(amount) > 10000 (verifies row filter not applied to pipeline identity)

SELECT
  COUNT(*)                                                            AS total_rows,
  SUM(CASE WHEN amt_to_monthly_spend IS NULL THEN 1 ELSE 0 END)      AS null_amt_to_monthly_spend,
  ROUND(100.0 * SUM(CASE WHEN amt_to_monthly_spend IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_amt_to_monthly_spend,
  MAX(amount)                                                         AS max_amount,
  CASE WHEN MAX(amount) > 10000 THEN 'PASS: row filter NOT applied'
       ELSE 'FAIL: row filter may be masking high amounts' END        AS row_filter_check,
  CASE WHEN 100.0 * SUM(CASE WHEN amt_to_monthly_spend IS NULL THEN 1 ELSE 0 END) / COUNT(*) < 1.0
       THEN 'PASS: column mask NOT applied'
       ELSE 'FAIL: column mask may be hiding avg_monthly_spend' END   AS column_mask_check
FROM workshop_guy_catalog.fe_bar_a0.txn_features_gold

total_rows,null_amt_to_monthly_spend,pct_null_amt_to_monthly_spend,max_amount,row_filter_check,column_mask_check
464679,33172,7.14,64435.14,PASS: row filter NOT applied,FAIL: column mask may be hiding avg_monthly_spend


In [0]:
%sql
-- (c) Leakage spot check: for 3 customers, show first 5 txns with txn_ts, txn_count_24h, is_new_device
-- Verify: first txn per customer should have txn_count_24h = 0 and is_new_device = 1
-- Subsequent txns should only count strictly earlier rows in their windows

WITH ranked AS (
  SELECT
    customer_id,
    txn_ts,
    txn_count_24h,
    is_new_device,
    secs_since_prev_txn,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY txn_ts, txn_id) AS rn
  FROM workshop_guy_catalog.fe_bar_a0.txn_features_gold
),
sample_customers AS (
  SELECT DISTINCT customer_id
  FROM workshop_guy_catalog.fe_bar_a0.txn_features_gold
  WHERE txn_count_24h > 0
  LIMIT 3
)
SELECT r.customer_id, r.txn_ts, r.txn_count_24h, r.is_new_device, r.secs_since_prev_txn
FROM ranked r
JOIN sample_customers s ON r.customer_id = s.customer_id
WHERE r.rn <= 5
ORDER BY r.customer_id, r.txn_ts

customer_id,txn_ts,txn_count_24h,is_new_device,secs_since_prev_txn
CUST-01025,2026-04-21T15:54:27.000Z,0,1,null
CUST-01025,2026-05-01T11:36:03.000Z,0,0,848496
CUST-01025,2026-05-08T10:25:21.000Z,0,1,600558
CUST-01025,2026-05-11T09:48:36.000Z,0,1,256995
CUST-01025,2026-05-13T09:33:05.000Z,0,1,171869
CUST-02033,2026-04-21T17:23:25.000Z,0,1,null
CUST-02033,2026-04-25T00:06:56.000Z,0,1,283411
CUST-02033,2026-04-27T12:44:27.000Z,0,1,218251
CUST-02033,2026-05-05T10:47:09.000Z,0,0,684162
CUST-02033,2026-05-06T13:25:19.000Z,0,1,95890


In [0]:
%sql
-- (d) Overall row count and fraud prevalence of txn_features_gold

SELECT
  COUNT(*)                                                      AS total_rows,
  SUM(CAST(is_fraud AS INT))                                    AS fraud_count,
  ROUND(100.0 * SUM(CAST(is_fraud AS INT)) / COUNT(*), 4)       AS fraud_prevalence_pct,
  COUNT(DISTINCT customer_id)                                    AS distinct_customers,
  COUNT(DISTINCT merchant_id)                                    AS distinct_merchants,
  MIN(txn_ts)                                                    AS earliest_txn,
  MAX(txn_ts)                                                    AS latest_txn
FROM workshop_guy_catalog.fe_bar_a0.txn_features_gold

total_rows,fraud_count,fraud_prevalence_pct,distinct_customers,distinct_merchants,earliest_txn,latest_txn
464679,10540,2.2682,10000,5000,2026-04-19T00:06:09.000Z,2026-10-15T23:55:43.000Z
